# STOCKSENSE: Model Explainability & Local Business Driver Breakdown
**NovaMart Retail Use Case (IntelliData 2026 Hackathon)**

This notebook demonstrates **Global Permutation Feature Importance** and **Local Manager-Friendly Driver Contributions** using STOCKSENSE's non-deep-learning explainability engine.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# Add root directory to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT_DIR))

from src.explain import ModelExplainer, sanity_test_explainability
from src.models.regression import FEATURE_COLS_NUM

## 1. Load Trained Pipeline Artifacts

In [ ]:
models_dir = ROOT_DIR / "models"
reg_pipe = joblib.load(models_dir / "demand_forecast_model.joblib")
print("Trained Pipeline Loaded Successfully.")

## 2. Local Row-Level Business Driver Breakdown
Translates raw feature sensitivities into plain-language business reasons with percentage impacts.

In [ ]:
explainer = ModelExplainer(reg_pipe, FEATURE_COLS_NUM)

# Create sample row
sample_row = pd.DataFrame([{
    "store_id": "S01", "product_id": "P001", "category": "Dairy", "store_type": "Express", "city": "Mumbai",
    "day_of_week": 4, "weekend_flag": 0, "month": 3, "week_no": 12, "day_of_month": 20,
    "festival_flag": 1, "holiday_flag": 0, "days_to_next_festival": 0, "days_since_last_festival": 10, "days_to_next_holiday": 5,
    "lag_1": 45.0, "lag_7": 40.0, "lag_14": 38.0, "lag_28": 35.0,
    "rolling_mean_7": 42.0, "rolling_std_7": 4.0, "rolling_max_7": 50.0, "rolling_min_7": 35.0,
    "rolling_mean_14": 40.0, "rolling_std_14": 5.0, "rolling_mean_28": 38.0,
    "recent_growth_rate": 1.15, "coefficient_of_variation_14": 0.125,
    "days_of_inventory": 2.5, "inventory_to_demand_ratio": 2.6, "reorder_gap": -10.0, "lead_time_demand": 126.0, "stock_cover_vs_lead_time": 0.83,
    "past_stockout_freq_30": 1.0, "days_since_last_stockout": 12,
    "discount_pct": 0.15, "price_to_mrp_ratio": 0.85, "price_change": -5.0, "promotion_flag": 1, "promo_days_last_7": 3, "days_since_last_promo": 0, "historical_promo_lift": 1.4,
    "perishability_flag": 1, "temp_c": 28.5, "rain_mm": 0.0, "temp_change": 1.5, "local_event": 0, "low_history_flag": 0
}])

explanation = explainer.explain_row_local(sample_row)
print("Predicted 7-Day Demand:", explanation["prediction"])
print("Top Drivers Breakdown:", explanation["reason_summary"])
for d in explanation["top_drivers"]:
    print(f" - {d['business_reason']}: {d['impact_pct']}%")

## 3. Explainability Sanity Verification

In [ ]:
sanity_test_explainability(explainer, sample_row)
print("Explainability engine sanity test PASSED!")